# Import BSF cell ranger output and save as Seurat object

In [ ]:
library_load <- suppressMessages(
    
    list(
        
        # Seurat 
        library(Seurat),
        
        # Data 
        library(dplyr), 
        
        # Pyhton compatibility
        library(reticulate)
        
    )
)

In [ ]:
# Configure reticulate 
use_condaenv(condaenv='p.3.8.12-FD20220623HSCT', conda="/nobackup/peer/fdeckert/miniconda3/bin/conda", required=NULL)
py_config()

In [ ]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20220623HSCT")

# Parameter settings

In [ ]:
# Seurat parameters 
min_cells    <- 0 # Filter out features which are only present in less than min_cells cells - default 3
min_features <- 0 # Filter out cells which have fewer than min_features features - default 200

# Files
so_file <- "data/object/raw.rds"
h5ad_file <- "data/object/raw.h5ad"

# Project folder

In [ ]:
sample_path <- "/nobackup/lab_bsf/projects/BSA_0738_Stary_amalgamation/OUT/COUNT/"
sample_dir <- list.dirs(path=sample_path, full.names=FALSE, recursive=FALSE) 

### HSCT samples

In [ ]:
sample_dir_1 <- sample_dir %>% 
    subset(., grepl("^S", .)) %>% 
    subset(., grepl("*transcriptome", .))
sample_path_1 <- paste0(sample_path, sample_dir_1)
patient_name_1 <- sapply(strsplit(sample_dir_1,"_"), `[`, 2) %>% substr(start=1 , stop=3)
time_point_1 <- sapply(strsplit(sample_dir_1,"_"), `[`, 3) %>% 
    gsub("baseline", "Baseline", .) %>% 
    gsub("day0", "Tx", .) %>% 
    gsub("day14", "D14", .) %>% 
    gsub("day100", "D100", .)

In [ ]:
sample_dir_2 <- sample_dir %>% 
    subset(., grepl("^HSCT", .)) %>% 
    subset(., grepl("*transcriptome", .))
sample_path_2 <- paste0(sample_path, sample_dir_2)
patient_name_2 <- sapply(strsplit(sample_dir_2,"_"), `[`, 2) %>% substr(start=1 , stop=3)
time_point_2 <- sapply(strsplit(sample_dir_2,"_"), `[`, 2) %>% substr(start=4 , stop=4) %>% 
    gsub("\\b1\\b", "Baseline", .) %>% 
    gsub("\\b2\\b", "Tx", .) %>% 
    gsub("\\b3\\b", "D14", .) %>% 
    gsub("\\b4\\b", "D100", .)

## GvD samples 

In [ ]:
sample_dir_3 <- sample_dir %>% 
    subset(., grepl("^GvD|^CoR_GvD|^CoR_GVD", .)) %>% 
    subset(., grepl("*transcriptome", .))
sample_path_3 <- paste0(sample_path, sample_dir_3)
patient_name_3 <- sapply(strsplit(sample_dir_3,"_"), `[`, 3) %>% substr(start=1 , stop=3)
time_point_3 <- rep("GVHD", length(sample_dir_3))

## Combine meta data

In [ ]:
meta <- data.frame(
  
    sample_path=c(sample_path_1, sample_path_2, sample_path_3), 
    sample_dir=c(sample_dir_1, sample_dir_2, sample_dir_3), 
    patient_name=c(patient_name_1, patient_name_2, patient_name_3), 
    time_point=c(time_point_1, time_point_2, time_point_3)
  
) %>% dplyr::arrange(patient_name)

In [ ]:
# digestion_protocol
meta$digestion_protocol <- ifelse(meta$sample_dir %in% sample_dir_1, "old", "new")

In [ ]:
# patient_id
meta$patient_id <- c(paste0("Patient_", 8), paste0("Patient_", 9), paste0("Patient_", 10), paste0("Patient_", 11), paste0("Patient_", 12), paste0("Patient_", rep(1, 3)), paste0("Patient_", rep(2, 2)), paste0("Patient_", rep(3, 4)), paste0("Patient_", rep(4, 4)), paste0("Patient_", rep(5, 3)), paste0("Patient_", rep(6, 3)), paste0("Patient_", rep(7, 3)))

In [ ]:
# sample_id 
meta$sample_name <- paste0(meta$patient_id, "_", meta$time_point)

In [ ]:
# Remove sample of chronic gvhd 
meta <- meta[!grepl("CoR_GVD_11", meta$sample_dir), ]

In [ ]:
# Write meta csv 
write.csv(meta, "meta.csv", row.names=FALSE)

# Import count matrix as Seurat object

In [ ]:
import_cnt <- function(sample_meta){
    
    message(paste("import:", paste0(sample_meta$sample_path, "/", "filtered_feature_bc_matrix")))
  
    # Read 10x data in as sparse matrix (sm)
    import_10x <- Read10X(data.dir=paste0(sample_meta$sample_path, "/", "filtered_feature_bc_matrix"))
    
    # Get count matrix 
    cnt <- import_10x[1][[1]]
    
    # Meta data 
    meta <- sapply(sample_meta, rep.int, times=ncol(cnt)) %>% data.frame()
    rownames(meta) <- colnames(cnt)
    
    # Transform sparse matrix to seurat object (so)
    so <- CreateSeuratObject(counts=cnt, meta.data=meta, min.cells=min_cells, min.features=min_features)
    
    # HTO demux information
    hto_demux_class <- read.csv(paste0(sample_meta$sample_path, "/", "HTO_demux.csv"), row.names=1)
    rownames(hto_demux_class) <- paste0(rownames(hto_demux_class), "-1")
    colnames(hto_demux_class) <- "hto_demux_class"
    so <- AddMetaData(so, hto_demux_class)
    
    # Set ident
    so$orig.ident <- NULL

    return(so)
  
}

so <- lapply(split(meta, f=meta$sample_path), import_cnt)

# Merge Seurat objects 
so <- merge(x=so[[1]], y=so[2:length(so)])

# Log normalize data 

In [ ]:
so <- NormalizeData(
    object=so, 
    assay="RNA", 
    normalization.method="LogNormalize", 
    scale.factor=10000
)

# Compute feature content

In [ ]:
# Compute MT content 
mt_genes <- rownames(so)[grep("^MT-", rownames(so))]
so[["pMT_RNA"]] <- PercentageFeatureSet(so, features=mt_genes)

# Compute Hemoglobin content 
hb_genes <- rownames(so)[grep("^HBA|^HBB", rownames(so))]
so[["pHB_RNA"]] <- PercentageFeatureSet(so, features=hb_genes)

# Compute Ribosomal content
rp_genes <- rownames(so)[grep("^RPL|^RPS", rownames(so))]
so[["pRP_RNA"]] <- PercentageFeatureSet(so, features=rp_genes)

# Complete meta data 

In [ ]:
so$hto_demux_class <- ifelse(so$hto_demux_class %in% c("HTO-blood", "HTO-Blood"), "Blood", so$hto_demux_class)
so$hto_demux_class <- ifelse(so$hto_demux_class %in% c("HTO-skin", "HTO-Skin"), "Skin", so$hto_demux_class)

# Save output file

In [ ]:
saveRDS(so, file=so_file)

In [ ]:
# Store data as h5ad 
adata <- import("anndata", as="adata", convert=FALSE)
pd <- import("pandas", as="pd", convert=FALSE)
np <- import("numpy", as="np", convert=FALSE)
    
# Transform dgCMatrix to sparse sc_sparse matrix
X <- GetAssayData(so, assay="RNA", slot="counts")    
X <- adata$AnnData(X=X)$X$T

# Combine 
adata <- adata$AnnData(X=X, obs=so@meta.data)
adata$var_names <- rownames(GetAssayData(so, assay="RNA", slot="counts"))

adata$raw <- adata

# Save
adata$write_h5ad(h5ad_file)

# Session info 

In [ ]:
sessionInfo()